# 08. Window Calculations & Resampling: Beginner Guide

### 📌 Overview
Master **08. Window Calculations & Resampling: Beginner Guide** with concise, zero-fluff bullet points and executable code on real Fintech records ([raw_transactions.csv](file:///data/raw_transactions.csv)).

### 📚 Key Concepts Covered in this Notebook:
- **Rolling & Expanding Windows**: Covers `.rolling().mean()` and `.expanding().sum()`.
- **Time-Series Resampling**: Covers `df.resample('ME').sum()`.
- **Shifting & Differencing**: Covers `.shift()`, `.diff()`, and `.pct_change()`.


In [1]:
# Setup imports & dataset loading
import pandas as pd
import numpy as np
import sys
import time
import os
import matplotlib.pyplot as plt

# Load raw transactions dataset
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
df = pd.read_csv(csv_path)
print(f"Pandas Version: {pd.__version__}")
print(f"Loaded raw_transactions.csv: {df.shape[0]} rows, {df.shape[1]} columns")

Pandas Version: 2.2.2
Loaded raw_transactions.csv: 15000 rows, 11 columns


### 🔹 Moving Average Windows with `.rolling()`
- **What it does:** Calculates rolling 7-day average transaction volume and spend.
- **Syntax:** `daily_ts['transaction_amount'].rolling(window=7, min_periods=1).mean()`
- **Operation:** `daily_ts = (`
- **Key Note:** Window calculations must be chained with an aggregation function (like `.mean()`, `.sum()`, or `.std()`) to compute the final windowed summary.

In [2]:
daily_ts = (
    df
    .assign(date=pd.to_datetime(df['transaction_date'], format='mixed', errors='coerce'))
    .dropna(subset=['date', 'transaction_amount'])
    .set_index('date')
    .sort_index()
    .resample('D')['transaction_amount'].sum()
)
rolling_7d = daily_ts.rolling(7, min_periods=1).mean()
print('7-Day Rolling Daily Total Spend Head:\n', rolling_7d.head(10))

7-Day Rolling Daily Total Spend Head:
 date
2025-01-01    28835.520000
2025-01-02    27827.710000
2025-01-03    26935.206667
2025-01-04    27447.230000
2025-01-05    26422.588000
2025-01-06    28900.385000
2025-01-07    28604.581429
2025-01-08    29687.974286
2025-01-09    29434.791429
2025-01-10    30197.855714
Freq: D, Name: transaction_amount, dtype: float64


### 🔹 Cumulative Running Totals with `.expanding()`
- **What it does:** Calculates lifetime running cumulative transaction volume.
- **Syntax:** `daily_ts.expanding().sum()`
- **Operation:** `cumulative_spend = daily_ts.expanding().sum()`
- **Key Note:** Inspect your data types early with `df.dtypes` to ensure numeric values weren't accidentally parsed as text.

In [3]:
cumulative_spend = daily_ts.expanding().sum()
print('Lifetime Expanding Cumulative Spend Head:\n', cumulative_spend.head())

Lifetime Expanding Cumulative Spend Head:
 date
2025-01-01     28835.52
2025-01-02     55655.42
2025-01-03     80805.62
2025-01-04    109788.92
2025-01-05    132112.94
Freq: D, Name: transaction_amount, dtype: float64


### 🔹 Frequency-Based Resampling with `.resample()`
- **What it does:** Aggregates transaction revenue by month-end (`'ME'`) and week (`'W'`).
- **Syntax:** `daily_ts.resample('ME').sum()`
- **Operation:** `monthly_spend = daily_ts.resample('ME').sum()`
- **Key Note:** Window calculations must be chained with an aggregation function (like `.mean()`, `.sum()`, or `.std()`) to compute the final windowed summary.

In [4]:
monthly_spend = daily_ts.resample('ME').sum()
print('Monthly Total Transaction Spend (resample ME):\n', monthly_spend)

Monthly Total Transaction Spend (resample ME):
 date
2025-01-31    881134.16
2025-02-28    827739.06
2025-03-31    860621.30
2025-04-30    895480.40
2025-05-31    941828.06
2025-06-30    875266.83
2025-07-31    899370.94
2025-08-31    905813.10
2025-09-30    866550.58
2025-10-31    830022.91
2025-11-30    865825.77
2025-12-31    849937.65
2026-01-31    781221.00
2026-02-28    765297.41
2026-03-31    844587.62
2026-04-30    808654.04
2026-05-31    410196.73
2026-06-30     24060.04
2026-07-31     36746.09
2026-08-31     25927.81
2026-09-30     34158.78
2026-10-31     32924.38
2026-11-30     27962.89
2026-12-31     35607.95
Freq: ME, Name: transaction_amount, dtype: float64


### 🔹 Lagging Time Series with `.shift()`
- **What it does:** Creates previous day's spend feature for time-series forecasting.
- **Syntax:** `daily_ts.shift(1)`
- **Operation:** `lagged_spend = daily_ts.shift(1)`
- **Key Note:** Ensure your column is converted to datetime with `pd.to_datetime()` before calling `.dt` properties like `.dt.year` or `.dt.day_name()`.

In [5]:
lagged_spend = daily_ts.shift(1)
print('Lagged Daily Spend (t-1) Head:\n', lagged_spend.head())

Lagged Daily Spend (t-1) Head:
 date
2025-01-01         NaN
2025-01-02    28835.52
2025-01-03    26819.90
2025-01-04    25150.20
2025-01-05    28983.30
Freq: D, Name: transaction_amount, dtype: float64


### 🔹 First-Order Differencing with `.diff()`
- **What it does:** Calculates day-over-day spend acceleration.
- **Syntax:** `daily_ts.diff(1)`
- **Operation:** `spend_diff = daily_ts.diff(1)`
- **Key Note:** Inspect your data types early with `df.dtypes` to ensure numeric values weren't accidentally parsed as text.

In [6]:
spend_diff = daily_ts.diff(1)
print('Day-over-Day Spend Delta Head:\n', spend_diff.head())

Day-over-Day Spend Delta Head:
 date
2025-01-01        NaN
2025-01-02   -2015.62
2025-01-03   -1669.70
2025-01-04    3833.10
2025-01-05   -6659.28
Freq: D, Name: transaction_amount, dtype: float64


### 🔹 Percentage Growth with `.pct_change()`
- **What it does:** Calculates daily growth rate in transaction volume.
- **Syntax:** `daily_ts.pct_change(1) * 100`
- **Operation:** `spend_growth = daily_ts.pct_change(1) * 100`
- **Key Note:** Inspect your data types early with `df.dtypes` to ensure numeric values weren't accidentally parsed as text.

In [7]:
spend_growth = daily_ts.pct_change(1) * 100
print('Daily Transaction Spend Growth (%):\n', spend_growth.dropna().head())

Daily Transaction Spend Growth (%):
 date
2025-01-02    -6.990059
2025-01-03    -6.225601
2025-01-04    15.240833
2025-01-05   -22.976266
2025-01-06    84.954905
Name: transaction_amount, dtype: float64


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data engineering questions explained with real examples.


### 🔍 Scenario: Q1: Rolling Volatility in Daily Revenue
- **Objective:** Q1: Rolling Volatility in Daily Revenue
- **Approach:** Calculate the 30-day rolling standard deviation of daily transaction revenue to monitor revenue stability.
- **Syntax:** `daily_ts.rolling(30).std()`

In [8]:
rolling_vol_30d = daily_ts.rolling(30).std()
print('30-Day Rolling Revenue Volatility (Std Dev):\n', rolling_vol_30d.dropna().head())

30-Day Rolling Revenue Volatility (Std Dev):
 date
2025-01-30    5487.849524
2025-01-31    5493.094169
2025-02-01    5495.220161
2025-02-02    5474.350681
2025-02-03    5534.801358
Freq: D, Name: transaction_amount, dtype: float64
